In [ ]:
"""Day 13 / Session 3 - Lab Assignment 1: Evaluation Metrics from Scratch (confusion matrix, PR/F1, ROC-AUC, thresholds)."""
import numpy as np
from scipy.stats import rankdata


def confusion_at(y_true, y_scores, threshold):
    y_true = np.asarray(y_true)
    pred = (np.asarray(y_scores) >= threshold).astype(int)
    return {
        "tp": int(np.sum((y_true == 1) & (pred == 1))),
        "fp": int(np.sum((y_true == 0) & (pred == 1))),
        "tn": int(np.sum((y_true == 0) & (pred == 0))),
        "fn": int(np.sum((y_true == 1) & (pred == 0))),
    }


def precision_recall_f1(tp, fp, tn, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}


def roc_auc(y_true, y_scores):
    """AUC = probability a random positive outranks a random negative (Mann-Whitney U / rank formula)."""
    y_true = np.asarray(y_true)
    ranks = rankdata(np.asarray(y_scores))          # ties get the average rank
    n_pos = int(np.sum(y_true == 1))
    n_neg = int(np.sum(y_true == 0))
    if n_pos == 0 or n_neg == 0:
        return 0.0
    u = ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2
    return float(u / (n_pos * n_neg))


def best_threshold(y_true, y_scores, thresholds):
    best_t, best_f1 = None, -1.0
    for t in thresholds:
        c = confusion_at(y_true, y_scores, t)
        f1 = precision_recall_f1(c["tp"], c["fp"], c["tn"], c["fn"])["f1"]
        if f1 > best_f1:  # strict '>' keeps the first threshold on ties
            best_t, best_f1 = t, f1
    return best_t